# Blocking medicine records across Canada, Europe and the United States

How many cross-source comparisons can we avoid while retaining the pairs reviewers labeled as matches? We compare lexical and embedding blockers on full source records and score their shortlists against reviewed pairs.

**Result.** Frozen Qwen top-5 retained 177/185 held-out positives. The selected matcher handoff uses **Qwen top-5 OR a four-character brand prefix**: 206,830 unique candidates. That union retained 183/185 positives in an exploratory follow-up on the same test labels.


## 1. Reproduce the experiment

Use Python 3.12 in the team repository. The reference data revision is `4283ce3c265e27cec737190820a8c0e6dc25dcf4`. From the repository root, install the project dependencies into your notebook environment:

```sh
python -m pip install -r requirements.txt
```

The embedding stage reads existing vectors if available. To encode from scratch, first download the pinned model to the Hugging Face cache using the same environment:

```sh
python -c "from huggingface_hub import snapshot_download; snapshot_download('Qwen/Qwen3-Embedding-0.6B', revision='97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3')"
```

Select this environment as the Jupyter kernel and **restart and run all cells in order**. The notebook runs offline after setup. It reads the three mapped XMLs, the EMA entity cross-reference, and the two gold splits. It writes diagnostic files and a timestamped handoff under `output/blocking/`. Candidate equality checks and published counts refer to the reference data and model.

**Reference environment.** PyDI 0.2.1, pandas 2.3.3, NumPy 2.1.3, scikit-learn 1.9.1, PyTorch 2.15.0, sentence-transformers 5.7.0 and transformers 4.57.6. The original run encoded on Apple MPS using float16; CPU encoding uses float32, so near-tied neighbors may vary. The export manifest records actual input, vector and output checksums and runtime versions.

In [1]:
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd

# Find the project if the notebook is anywhere inside the repository.
folder = Path.cwd()
repo = next(
    (p for p in (folder, *folder.parents) if (p / "data/4_translated_mapforce").is_dir()),
    None,
)
if repo is None:
    raise FileNotFoundError("Launch Jupyter from the project root or its notebooks/ directory.")

xml_dir = repo / "data/4_translated_mapforce"
gold_dir = repo / "data/5_golden_standard_pair/golden_standard_pair_complete"
gold = pd.read_csv(gold_dir / "gs_pair_train_val.csv", dtype=str, encoding="utf-8-sig")

ids = {}
for source in ("DPD", "EMA", "FDA"):
    records = ET.parse(xml_dir / f"{source.lower()}_medicines.xml").getroot()
    medicines = records.findall("medicine")
    ids[source] = {record.findtext("id") for record in medicines}
    assert all(ids[source]) and len(ids[source]) == len(medicines), f"Invalid IDs: {source}"

rows = []
for (left, right), pairs in gold.groupby(["source1", "source2"]):
    found = pairs["id1"].isin(ids[left]) & pairs["id2"].isin(ids[right])
    rows.append({
        "Source pair": f"{left}–{right}",
        "Possible comparisons": len(ids[left]) * len(ids[right]),
        "Labeled pairs": len(pairs),
        "Both IDs found": int(found.sum()),
    })

display(pd.DataFrame(rows))

,Source pair,Possible comparisons,Labeled pairs,Both IDs found
0,DPD–EMA,110252128,850,850
1,DPD–FDA,340374300,653,653
2,EMA–FDA,21647400,1470,0


### Align EMA identifiers

Some reviewed rows use the EMA regulatory product number. The prepared entity table links each contributing product number to the current grouped medicine ID. Translate these within-source IDs before evaluating cross-source matches. Preserve each original ID for review.

In [2]:
ema = pd.read_csv(repo / "data/2_processed/ema_entities.csv", dtype=str)

# One current medicine can have several original EMA product numbers.
lookup = ema.assign(old_number=ema["ema_product_numbers"].str.split(";")).explode("old_number")
lookup["old_id"] = "EMA_" + lookup["old_number"].str.strip()
lookup["current_id"] = "EMA_" + lookup["medicine_id"]

mapping = lookup[["old_id", "current_id"]].drop_duplicates()
assert mapping["old_id"].is_unique, "An old EMA ID maps to multiple medicines."
ema_ids = mapping.set_index("old_id")["current_id"]

dev = gold.copy()
for side in ("1", "2"):
    source, key = "source" + side, "id" + side
    dev[key + "_original"] = dev[key]
    needs_lookup = dev[source].eq("EMA") & ~dev[key].isin(ids["EMA"])
    dev.loc[needs_lookup, key] = dev.loc[needs_lookup, key].map(ema_ids)
    print(f"Side {side}: {needs_lookup.sum():,} EMA IDs needed translation")

dev["both_ids_found"] = [
    a in ids[s1] and b in ids[s2]
    for a, s1, b, s2 in zip(dev.id1, dev.source1, dev.id2, dev.source2)
]
display(dev.groupby(["source1", "source2"])["both_ids_found"].agg(["size", "sum"]))
assert dev["both_ids_found"].all(), "Resolve unmapped development IDs before evaluation."

Side 1: 1,470 EMA IDs needed translation
Side 2: 0 EMA IDs needed translation


size   sum
source1 source2            
DPD     EMA       850   850
        FDA       653   653
EMA     FDA      1470  1470

### Resolve repeated development pairs

Group by both source names and both current IDs. Count agreeing repeats once, retaining their hard flag if any copy was sampled as hard. Set aside every row of a pair if its match labels conflict.

In [3]:
pair_key = ["source1", "id1", "source2", "id2"]

# Keep every row belonging to a repeated pair so we can inspect both labels.
repeats = dev.loc[dev.duplicated(pair_key, keep=False)].copy()
groups = repeats.groupby(pair_key)["match"].agg(rows="size", distinct_labels="nunique")

print(f"Repeated pairs:           {len(groups)}")
print(f"  With agreeing labels:   {groups['distinct_labels'].eq(1).sum()}")
print(f"  With conflicting labels:{groups['distinct_labels'].gt(1).sum():>4}")
print(f"Original rows involved:   {len(repeats)}")

# Inspect the conflicts side by side.
conflicts = repeats.groupby(pair_key).filter(lambda g: g["match"].nunique() > 1)
display(conflicts.sort_values(pair_key)[["id1_original", "id1", "id2", "match"]])

Repeated pairs:           16
  With agreeing labels:   13
  With conflicting labels:   3
Original rows involved:   32


,id1_original,id1,id2,match
913,EMA_EMEA/H/C/003926,EMA_ARIPIPRAZOLE MYLAN|ARIPIPRAZOLE,FDA_ARIPIPRAZOLE|ARIPIPRAZOLE,1
1761,EMA_EMEA/H/C/004236,EMA_ARIPIPRAZOLE MYLAN|ARIPIPRAZOLE,FDA_ARIPIPRAZOLE|ARIPIPRAZOLE,0
1266,EMA_EMEA/H/C/000612,EMA_IONSYS|FENTANYL HYDROCHLORIDE,FDA_IONSYS|FENTANYL HYDROCHLORIDE,1
2158,EMA_EMEA/H/C/002715,EMA_IONSYS|FENTANYL HYDROCHLORIDE,FDA_IONSYS|FENTANYL HYDROCHLORIDE,0
363,EMA_EMEA/H/C/001240,EMA_VIBATIV|TELAVANCIN,FDA_VIBATIV|TELAVANCIN,1
809,EMA_EMEA/H/C/000864,EMA_VIBATIV|TELAVANCIN,FDA_VIBATIV|TELAVANCIN,0


In [4]:
# Mark every row whose pair has more than one match label.
labels_per_pair = dev.groupby(pair_key)["match"].transform("nunique")
has_conflict = labels_per_pair.gt(1)

conflicts_for_review = dev.loc[has_conflict].copy()

# Among the remaining rows, keep one copy per pair.
# If a pair was sampled as both ordinary and hard, retain its hard flag.
dev_eval = (
    dev.loc[~has_conflict]
    .sort_values("hard", ascending=False, kind="stable")
    .drop_duplicates(pair_key)
    .copy()
)

assert not dev_eval.duplicated(pair_key).any()
print(f"Conflicting rows set aside: {len(conflicts_for_review):,}")
print(f"Extra agreeing rows removed: {len(dev) - len(conflicts_for_review) - len(dev_eval):,}")

summary = (
    dev_eval.assign(positive=dev_eval["match"].astype(int))
    .groupby(["source1", "source2"])
    .agg(eligible_pairs=("match", "size"), known_matches=("positive", "sum"))
)
display(summary)
print(summary.sum())

Conflicting rows set aside: 6
Extra agreeing rows removed: 13


eligible_pairs  known_matches
source1 source2                               
DPD     EMA                 850            352
        FDA                 653            211
EMA     FDA                1451            475

eligible_pairs    2954
known_matches     1038
dtype: int64


The audit leaves 2,954 distinct development pairs with 1,038 positives. Thirteen agreeing repeats were collapsed and six rows from three conflicting pairs were set aside. `dev_eval` supplies the evaluation labels; the complete XML records supply the blockers' inputs.

## 2. Development experiments

**Pair completeness** is the fraction of eligible known positives present in the candidate list. **Reduction ratio** equals one minus the number of unique candidates divided by the product of the two source sizes. Select the smallest tested shortlist with completeness above 97% on each development source pair. Report counts alongside percentages because the number of known positives differs by pair.

### Brand-prefix baseline

Normalize brand text with the team function, retain alphanumerics, and compare the first three or four characters. Empty keys produce no candidates. Runtime covers blocker construction, candidate enumeration and deduplication.

In [5]:
import sys
from time import perf_counter

from PyDI.entitymatching.blocking import StandardBlocker

# Reuse the team's normalization function.
sys.path.insert(0, str(repo / "src/golden_standard"))
from gold_standard_pair_gen import normalize_text

def brand_key(brand):
    normalized = normalize_text(brand)
    return "".join(c for c in normalized if c.isalnum())

# Build prefix keys from the full XML sources.
brand_data = {}
for source in ("DPD", "EMA", "FDA"):
    root = ET.parse(xml_dir / f"{source.lower()}_medicines.xml").getroot()
    records = [
        (m.findtext("id"), brand_key(m.findtext("brandName") or ""))
        for m in root.findall("medicine")
    ]
    frame = pd.DataFrame(records, columns=["id", "brand_key"])
    brand_data[source] = frame.loc[frame["brand_key"].ne("")]

# Generate candidates, then check how many known development matches survived.
results = []
for length in (3, 4):
    keyed = {s: frame.assign(prefix=frame["brand_key"].str[:length])
             for s, frame in brand_data.items()}
    for left, right in [("DPD", "EMA"), ("DPD", "FDA"), ("EMA", "FDA")]:
        start = perf_counter()
        blocker = StandardBlocker(
            keyed[left], keyed[right],
            on=["prefix"],
            id_column="id",
            output_dir=str(repo / "output" / "blocking" / f"prefix{length}" / f"{left}_{right}"),
        )
        candidates = blocker.materialize()[["id1", "id2"]].drop_duplicates()
        seconds = perf_counter() - start

        positives = dev_eval.loc[
            dev_eval["source1"].eq(left)
            & dev_eval["source2"].eq(right)
            & dev_eval["match"].astype(int).eq(1),
            ["id1", "id2"],
        ]
        retained = len(positives.merge(candidates, on=["id1", "id2"], validate="one_to_one"))
        possible = len(ids[left]) * len(ids[right])

        results.append({
            "Source pair": f"{left}–{right}",
            "Prefix length": length,
            "Candidates": len(candidates),
            "Known matches": len(positives),
            "Retained": retained,
            "Missed": len(positives) - retained,
            "Pair completeness": retained / len(positives),
            "Reduction ratio": 1 - len(candidates) / possible,
            "Blocking seconds": seconds,
        })

prefix_results = pd.DataFrame(results)
display(prefix_results.style.format({
    "Pair completeness": "{:.2%}",
    "Reduction ratio": "{:.4%}",
    "Blocking seconds": "{:.2f}",
}))

/Users/anatolelobenko/Desktop/WDI2026/.venv-pydi/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Source pair,Prefix length,Candidates,Known matches,Retained,Missed,Pair completeness,Reduction ratio,Blocking seconds
0,DPD–EMA,3,96865,352,313,39,88.92%,99.9121%,0.13
1,DPD–FDA,3,486376,211,210,1,99.53%,99.8571%,0.28
2,EMA–FDA,3,23159,475,468,7,98.53%,99.8930%,0.04
3,DPD–EMA,4,16398,352,313,39,88.92%,99.9851%,0.11
4,DPD–FDA,4,133069,211,210,1,99.53%,99.9609%,0.16
5,EMA–FDA,4,4456,475,468,7,98.53%,99.9794%,0.03


Prefixes of three and four characters retained the same 991 development matches. Four characters reduced the total from 606,400 to 153,923 candidates. The four-character baseline misses 39 of the 352 DPD–EMA positives, motivating a second lexical pass.

### Brand trigrams and the cost of common fragments

Count each distinct three-character fragment once per record. A fragment matching *a* left and *b* right records contributes *a × b* possible pairs. First compare caps of 500 and 1,000 on DPD–EMA. Then apply the cap of 1,000 separately on each source pair and evaluate the deduplicated union with prefix-4.

In [6]:
from collections import Counter
from PyDI.entitymatching.blocking import TokenBlocker

left, right = brand_data["DPD"], brand_data["EMA"]
cap = 500  # Maximum DPD × EMA comparisons allowed for one shared trigram.

def trigrams(name):
    return {name[i:i + 3] for i in range(len(name) - 2)}

left_freq = Counter(t for name in left["brand_key"] for t in trigrams(name))
right_freq = Counter(t for name in right["brand_key"] for t in trigrams(name))
allowed = {
    t for t in left_freq.keys() & right_freq.keys()
    if left_freq[t] * right_freq[t] <= cap
}

blocker = TokenBlocker(
    left, right,
    column="brand_key",
    id_column="id",
    tokenizer=lambda name: sorted(trigrams(name) & allowed),
    output_dir=str(repo / "output" / "blocking" / "trigram500" / "DPD_EMA"),
)
trigram_candidates = blocker.materialize()[["id1", "id2"]].drop_duplicates()

# Check which individual gold matches each method retained.
gold = dev_eval.loc[
    dev_eval["source1"].eq("DPD")
    & dev_eval["source2"].eq("EMA")
    & dev_eval["match"].astype(int).eq(1),
    ["id1", "id2"],
].copy()

gold["dpd_brand"] = gold["id1"].map(left.set_index("id")["brand_key"])
gold["ema_brand"] = gold["id2"].map(right.set_index("id")["brand_key"])
gold["prefix4"] = gold["dpd_brand"].str[:4].eq(gold["ema_brand"].str[:4])
gold["trigram"] = pd.MultiIndex.from_frame(gold[["id1", "id2"]]).isin(
    pd.MultiIndex.from_frame(trigram_candidates)
)

prefix_row = prefix_results.loc[
    prefix_results["Source pair"].eq("DPD–EMA") & prefix_results["Prefix length"].eq(4)
].iloc[0]
assert gold["prefix4"].sum() == prefix_row["Retained"]

recovered = gold.loc[~gold["prefix4"] & gold["trigram"]]
prefix_only = gold.loc[gold["prefix4"] & ~gold["trigram"]]

comparison = pd.DataFrame({
    "Method": ["4-character prefix", f"Brand trigrams (cap {cap})"],
    "Candidates": [prefix_row["Candidates"], len(trigram_candidates)],
    "Known matches retained": [gold["prefix4"].sum(), gold["trigram"].sum()],
})
comparison["Pair completeness"] = comparison["Known matches retained"] / len(gold)
display(comparison.style.hide(axis="index").format({
    "Candidates": "{:,}", "Pair completeness": "{:.2%}",
}))

print(f"Prefix misses recovered by trigrams: {len(recovered)} of {(~gold['prefix4']).sum()}")
print(f"Matches kept by prefix but missed by trigrams: {len(prefix_only)}")
print(f"Known matches retained by either method: {(gold['prefix4'] | gold['trigram']).sum()} of {len(gold)}")
display(recovered[["dpd_brand", "ema_brand"]].head(10))

Method,Candidates,Known matches retained,Pair completeness
4-character prefix,"16,398",313,88.92%
Brand trigrams (cap 500),"240,915",282,80.11%


Prefix misses recovered by trigrams: 21 of 39
Matches kept by prefix but missed by trigrams: 52
Known matches retained by either method: 334 of 352


,dpd_brand,ema_brand
118,tevatemozolomide,temozolomideteva
217,sandozlevodopacarbidopaentacapone,corbiltapreviouslylevodopacarbidopaentacapones...
219,sandozaripiprazoletablet,aripiprazolesandoz
272,actemra,roactemra
463,sandozrivastigminepatch5,rivastigminesandoz
487,tevadimethylfumarate,dimethylfumarateteva
564,sandozaripiprazole,aripiprazolesandoz
643,tevapramipexole,pramipexoleteva
993,xpovio,nexpovio
1070,sandozrivastigmine,rivastigminesandoz


#### Relax the DPD–EMA trigram cap

Compare the broader fragment set with the same development positives and count newly retrieved pairs.

In [7]:
cap_new = 1000
allowed_1000 = {
    t for t in left_freq.keys() & right_freq.keys()
    if left_freq[t] * right_freq[t] <= cap_new
}

blocker_1000 = TokenBlocker(
    left, right,
    column="brand_key",
    id_column="id",
    tokenizer=lambda name: sorted(trigrams(name) & allowed_1000),
    output_dir=str(repo / "output" / "blocking" / "trigram1000" / "DPD_EMA"),
)
candidates_1000 = blocker_1000.materialize()[["id1", "id2"]].drop_duplicates()

gold_pairs = pd.MultiIndex.from_frame(gold[["id1", "id2"]])
hit_1000 = pd.Series(
    gold_pairs.isin(pd.MultiIndex.from_frame(candidates_1000)),
    index=gold.index,
)

# Raising the cap should preserve every previous trigram hit.
assert hit_1000[gold["trigram"]].all()

missed_before = ~gold["prefix4"] & ~gold["trigram"]
rescued = gold.loc[missed_before & hit_1000]

print(f"Trigram candidates: {len(trigram_candidates):,} → {len(candidates_1000):,}")
print(f"Additional candidates: {len(candidates_1000) - len(trigram_candidates):,}")
print(f"Trigram-only known matches: {gold['trigram'].sum()} → {hit_1000.sum()}")
print(f"Recovered from the {missed_before.sum()} missed by both previous rules: {len(rescued)}")
print(f"Prefix OR cap-1000 trigrams: {(gold['prefix4'] | hit_1000).sum()} / {len(gold)}")
display(rescued[["dpd_brand", "ema_brand"]])

Trigram candidates: 240,915 → 431,353
Additional candidates: 190,438
Trigram-only known matches: 282 → 323
Recovered from the 18 missed by both previous rules: 12
Prefix OR cap-1000 trigrams: 346 / 352


,dpd_brand,ema_brand
301,mylanpregabalin,pregabalinviatrispreviouslypregabalinmylan
389,ratiopioglitazone,pioglitazoneratio
651,tevairbesartanhctz,irbesartanhydrochlorothiazideteva
1198,sandozpregabalin,pregabalinsandoz
1786,tevaolanzapineodt,olanzapineteva
2019,tevalevetiracetam,levetiracetamteva
2131,mylanolanzapine,olanzapineviatrispreviouslyolanzapinemylan
2293,sandozduloxetine,duloxetinesandoz
2390,trintellix,brintellix
2517,tevaolanzapineod,olanzapineteva


#### Apply the lexical rules across all three source pairs

In [8]:
from collections import Counter
from itertools import combinations
from PyDI.entitymatching.blocking import StandardBlocker, TokenBlocker

cap = 1000
sources = ["DPD", "EMA", "FDA"]

def brand_trigrams(name):
    return {name[i:i + 3] for i in range(len(name) - 2)}

frames = {
    source: brand_data[source].assign(prefix4=brand_data[source]["brand_key"].str[:4])
    for source in sources
}
frequencies = {
    source: Counter(t for name in frames[source]["brand_key"] for t in brand_trigrams(name))
    for source in sources
}

def candidate_index(blocker):
    pairs = blocker.materialize()[["id1", "id2"]]
    return pd.MultiIndex.from_frame(pairs).unique()

rows = []
for left, right in combinations(sources, 2):
    pair_dir = repo / "output" / "blocking" / f"{left}_{right}"

    prefix_pairs = candidate_index(StandardBlocker(
        frames[left], frames[right],
        on=["prefix4"],
        id_column="id",
        output_dir=str(pair_dir / "prefix4"),
    ))

    # Calculate the cap separately for each source pair.
    left_freq, right_freq = frequencies[left], frequencies[right]
    allowed = {
        t for t in left_freq.keys() & right_freq.keys()
        if left_freq[t] * right_freq[t] <= cap
    }
    trigram_pairs = candidate_index(TokenBlocker(
        frames[left], frames[right],
        column="brand_key",
        id_column="id",
        tokenizer=lambda name: sorted(brand_trigrams(name) & allowed),
        output_dir=str(pair_dir / "trigram1000"),
    ))

    combined_pairs = prefix_pairs.union(trigram_pairs, sort=False)

    positives = dev_eval.loc[
        dev_eval["source1"].eq(left)
        & dev_eval["source2"].eq(right)
        & dev_eval["match"].astype(int).eq(1),
        ["id1", "id2"],
    ]
    gold_pairs = pd.MultiIndex.from_frame(positives)
    possible = len(ids[left]) * len(ids[right])

    for method, candidates in [
        ("Prefix 4", prefix_pairs),
        ("Trigrams (cap 1,000)", trigram_pairs),
        ("Prefix 4 OR trigrams", combined_pairs),
    ]:
        retained = int(gold_pairs.isin(candidates).sum())
        rows.append({
            "Source pair": f"{left}–{right}",
            "Method": method,
            "Unique candidates": len(candidates),
            "Known matches": len(gold_pairs),
            "Retained": retained,
            "Missed": len(gold_pairs) - retained,
            "Pair completeness": retained / len(gold_pairs),
            "Reduction ratio": 1 - len(candidates) / possible,
        })

combined_results = pd.DataFrame(rows)
display(combined_results.style.hide(axis="index").format({
    "Unique candidates": "{:,}",
    "Pair completeness": "{:.2%}",
    "Reduction ratio": "{:.4%}",
}))

Source pair,Method,Unique candidates,Known matches,Retained,Missed,Pair completeness,Reduction ratio
DPD–EMA,Prefix 4,"16,398",352,313,39,88.92%,99.9851%
DPD–EMA,"Trigrams (cap 1,000)","431,353",352,323,29,91.76%,99.6088%
DPD–EMA,Prefix 4 OR trigrams,"443,207",352,346,6,98.30%,99.5980%
DPD–FDA,Prefix 4,"133,069",211,210,1,99.53%,99.9609%
DPD–FDA,"Trigrams (cap 1,000)","544,036",211,172,39,81.52%,99.8402%
DPD–FDA,Prefix 4 OR trigrams,"663,705",211,210,1,99.53%,99.8050%
EMA–FDA,Prefix 4,"4,456",475,468,7,98.53%,99.9794%
EMA–FDA,"Trigrams (cap 1,000)","261,601",475,472,3,99.37%,98.7915%
EMA–FDA,Prefix 4 OR trigrams,"262,625",475,474,1,99.79%,98.7868%


### Qwen embeddings and nearest-neighbor retrieval

Encode the full brand and sorted active-substance names once per medicine without fine-tuning. Query from the smaller source into the larger. A single cosine top-20 search supplies the top-5, top-10 and top-20 comparisons. Source IDs are reordered to agree with the gold-file orientation.

In the original cold run, Qwen encoding took 496.9 seconds across all sources and the three searches took 9.7 seconds. Cache-based reruns skip encoding. Search timing includes index construction and candidate materialization.

In [9]:
from pathlib import Path
from itertools import combinations
from time import perf_counter
import hashlib
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from PyDI.entitymatching.blocking import EmbeddingBlocker

import os
from huggingface_hub import snapshot_download

MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"
MODEL_REVISION = "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3"
device = "mps" if torch.backends.mps.is_available() else "cpu"
model = None  # Loaded only on a cache miss.
cache_root = Path(os.environ.get("WDI_EMBEDDING_CACHE", repo.parent / ".embedding-cache"))
cache_dir = cache_root / MODEL_REVISION
cache_dir.mkdir(parents=True, exist_ok=True)
vector_paths = {}
frames, vectors = {}, {}

for source in ("DPD", "EMA", "FDA"):
    records = []
    for med in ET.parse(xml_dir / f"{source.lower()}_medicines.xml").getroot().findall("medicine"):
        brand = (med.findtext("brandName") or "").strip()
        ingredients = sorted({
            node.text.strip()
            for node in med.findall("activeSubstances/activeSubstance")
            if node.text and node.text.strip()
        }, key=str.casefold)
        text = f"Brand: {brand}\nActive substances: {'; '.join(ingredients)}"
        records.append((med.findtext("id"), text))

    frame = pd.DataFrame(records, columns=["id", "text"])
    fingerprint = hashlib.sha256(frame.to_csv(index=False).encode()).hexdigest()[:16]
    cache_file = cache_dir / f"{source}-{fingerprint}.npy"

    if cache_file.exists():
        embeddings = np.load(cache_file, allow_pickle=False)
        print(f"{source}: loaded {len(frame):,} cached vectors", flush=True)
    else:
        print(f"{source}: encoding {len(frame):,} medicines...", flush=True)
        if model is None:
            model_dir = snapshot_download(MODEL_ID, revision=MODEL_REVISION, local_files_only=True)
            model = SentenceTransformer(
                model_dir, device=device, local_files_only=True,
                model_kwargs={"dtype": torch.float16 if device == "mps" else torch.float32},
            )
            model.tokenizer.padding_side = "left"
            model.eval()
        start = perf_counter()
        embeddings = model.encode(
            frame["text"].tolist(),
            batch_size=32,
            prompt="",
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=True,
        ).astype(np.float32)
        temp_file = cache_file.with_suffix(".tmp.npy")
        np.save(temp_file, embeddings)
        temp_file.replace(cache_file)
        print(f"{source}: encoded in {perf_counter() - start:.1f} seconds", flush=True)

    vector_paths[source] = cache_file
    assert embeddings.shape == (len(frame), 1024)
    assert np.isfinite(embeddings).all()
    assert np.allclose(np.linalg.norm(embeddings, axis=1), 1, atol=0.002)
    frames[source], vectors[source] = frame, embeddings

# Retrieve top 20 once per source pair. Score the first 5, 10 and 20.
# Query from the smaller source; keep gold IDs in their original source order.
rows = []
qwen_ranked = {}

for left, right in combinations(("DPD", "EMA", "FDA"), 2):
    query, target = sorted((left, right), key=lambda s: len(frames[s]))
    start = perf_counter()

    blocker = EmbeddingBlocker(
        frames[query], frames[target],
        text_cols=["text"],
        id_column="id",
        left_embeddings=vectors[query],
        right_embeddings=vectors[target],
        index_backend="sklearn",
        metric="cosine",
        top_k=20,
        threshold=0.0,  # Lowest cosine cutoff supported by this PyDI version.
        query_batch_size=128,
        output_dir=str(repo / "output" / "blocking" / "qwen" / f"{left}_{right}"),
    )
    ranked = blocker.materialize()[["id1", "id2"]].copy()
    search_seconds = perf_counter() - start

    # PyDI emits neighbors in decreasing similarity order for each query ID.
    ranked["rank"] = ranked.groupby("id1", sort=False).cumcount() + 1
    if query != left:
        ranked = ranked.rename(columns={"id1": "id2", "id2": "id1"})
    ranked = ranked[["id1", "id2", "rank"]]
    qwen_ranked[(left, right)] = ranked

    positives = dev_eval.loc[
        dev_eval["source1"].eq(left)
        & dev_eval["source2"].eq(right)
        & dev_eval["match"].astype(int).eq(1),
        ["id1", "id2"],
    ]
    gold_pairs = pd.MultiIndex.from_frame(positives)
    possible = len(frames[left]) * len(frames[right])

    print(f"{query} → {target}: top-20 search took {search_seconds:.1f} seconds")
    for k in (5, 10, 20):
        candidates = pd.MultiIndex.from_frame(
            ranked.loc[ranked["rank"].le(k), ["id1", "id2"]]
        ).unique()
        retained = int(gold_pairs.isin(candidates).sum())
        rows.append({
            "Source pair": f"{left}–{right}",
            "Query → target": f"{query} → {target}",
            "Top k": k,
            "Unique candidates": len(candidates),
            "Known matches": len(gold_pairs),
            "Retained": retained,
            "Missed": len(gold_pairs) - retained,
            "Pair completeness": retained / len(gold_pairs),
            "Reduction ratio": 1 - len(candidates) / possible,
        })

qwen_results = pd.DataFrame(rows)
display(qwen_results.style.hide(axis="index").format({
    "Unique candidates": "{:,}",
    "Pair completeness": "{:.2%}",
    "Reduction ratio": "{:.4%}",
}))

DPD: loaded 41,636 cached vectors
EMA: loaded 2,648 cached vectors
FDA: loaded 8,175 cached vectors


/Users/anatolelobenko/Desktop/WDI2026/.venv-pydi/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/anatolelobenko/Desktop/WDI2026/.venv-pydi/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/anatolelobenko/Desktop/WDI2026/.venv-pydi/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/anatolelobenko/Desktop/WDI2026/.venv-pydi/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/anatolelobenko/Desktop/WDI2026/.venv-pydi/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/anatolelobenko/Desktop/WDI2026/.venv-pydi/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid value 

EMA → DPD: top-20 search took 2.5 seconds


/Users/anatolelobenko/Desktop/WDI2026/.venv-pydi/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/anatolelobenko/Desktop/WDI2026/.venv-pydi/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/anatolelobenko/Desktop/WDI2026/.venv-pydi/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/anatolelobenko/Desktop/WDI2026/.venv-pydi/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/anatolelobenko/Desktop/WDI2026/.venv-pydi/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/anatolelobenko/Desktop/WDI2026/.venv-pydi/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid value 

FDA → DPD: top-20 search took 6.8 seconds


/Users/anatolelobenko/Desktop/WDI2026/.venv-pydi/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/anatolelobenko/Desktop/WDI2026/.venv-pydi/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/anatolelobenko/Desktop/WDI2026/.venv-pydi/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/anatolelobenko/Desktop/WDI2026/.venv-pydi/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/anatolelobenko/Desktop/WDI2026/.venv-pydi/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/anatolelobenko/Desktop/WDI2026/.venv-pydi/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid value 

EMA → FDA: top-20 search took 1.1 seconds


Source pair,Query → target,Top k,Unique candidates,Known matches,Retained,Missed,Pair completeness,Reduction ratio
DPD–EMA,EMA → DPD,5,"13,240",352,346,6,98.30%,99.9880%
DPD–EMA,EMA → DPD,10,"26,480",352,349,3,99.15%,99.9760%
DPD–EMA,EMA → DPD,20,"52,960",352,349,3,99.15%,99.9520%
DPD–FDA,FDA → DPD,5,"40,875",211,210,1,99.53%,99.9880%
DPD–FDA,FDA → DPD,10,"81,750",211,211,0,100.00%,99.9760%
DPD–FDA,FDA → DPD,20,"163,500",211,211,0,100.00%,99.9520%
EMA–FDA,EMA → FDA,5,"13,240",475,465,10,97.89%,99.9388%
EMA–FDA,EMA → FDA,10,"26,480",475,468,7,98.53%,99.8777%
EMA–FDA,EMA → FDA,20,"52,960",475,470,5,98.95%,99.7554%


### What does the embedding pass recover?

Compare the identities of the DPD–EMA misses. Equal counts can conceal different missed pairs. Test prefix-4 as a lower-cost addition to Qwen before extending the union to every source pair.

In [10]:
# Compare the actual missed DPD–EMA pairs using the existing notebook results.
check = dev_eval.loc[
    dev_eval["source1"].eq("DPD")
    & dev_eval["source2"].eq("EMA")
    & dev_eval["match"].astype(int).eq(1),
    ["id1", "id2"],
].copy()

check["DPD brand"] = check["id1"].map(brand_data["DPD"].set_index("id")["brand_key"])
check["EMA brand"] = check["id2"].map(brand_data["EMA"].set_index("id")["brand_key"])
pairs = pd.MultiIndex.from_frame(check[["id1", "id2"]])

prefix_hit = check["DPD brand"].str[:4].eq(check["EMA brand"].str[:4])
trigram_hit = pairs.isin(pd.MultiIndex.from_frame(candidates_1000[["id1", "id2"]]))
check["Missed by prefix + trigrams"] = ~(prefix_hit | trigram_hit)

ranked = qwen_ranked[("DPD", "EMA")]
top5 = ranked.loc[ranked["rank"].le(5), ["id1", "id2"]]
check["Missed by Qwen top-5"] = ~pairs.isin(pd.MultiIndex.from_frame(top5))

lex = check["Missed by prefix + trigrams"]
qwen = check["Missed by Qwen top-5"]
assert len(check) == 352 and lex.sum() == 6 and qwen.sum() == 6

print(f"Missed by both:           {(lex & qwen).sum()} of 6")
print(f"Missed only by lexical:   {(lex & ~qwen).sum()}")
print(f"Missed only by Qwen:      {(qwen & ~lex).sum()}")
print(f"Identical missed pairs:   {lex.equals(qwen)}")

display(
    check.loc[lex | qwen]
    .sort_values(["Missed by prefix + trigrams", "Missed by Qwen top-5"], ascending=False)
    .reset_index(drop=True)
)

Missed by both:           0 of 6
Missed only by lexical:   6
Missed only by Qwen:      6
Identical missed pairs:   False


,id1,id2,DPD brand,EMA brand,Missed by prefix + trigrams,Missed by Qwen top-5
0,DPD_TEVA ENTACAPONE|ENTACAPONE,EMA_ENTACAPONE TEVA|ENTACAPONE,tevaentacapone,entacaponeteva,True,False
1,DPD_TEVA MYCOPHENOLATE|MYCOPHENOLATE MOFETIL,EMA_MYCOPHENOLATE MOFETIL TEVA|MYCOPHENOLATE M...,tevamycophenolate,mycophenolatemofetilteva,True,False
2,DPD_SANDOZ CLOPIDOGREL|CLOPIDOGREL,EMA_CLOPIDOGREL SANDOZ|CLOPIDOGREL,sandozclopidogrel,clopidogrelsandoz,True,False
3,DPD_TEVA TOPOTECAN|TOPOTECAN,EMA_TOPOTECAN TEVA|TOPOTECAN,tevatopotecan,topotecanteva,True,False
4,DPD_TEVA CAPECITABINE|CAPECITABINE,EMA_CAPECITABINE TEVA|CAPECITABINE,tevacapecitabine,capecitabineteva,True,False
5,DPD_TEVA IMATINIB|IMATINIB,EMA_IMATINIB TEVA|IMATINIB,tevaimatinib,imatinibteva,True,False
6,DPD_SUSTIVA TABLETS|EFAVIRENZ,EMA_SUSTIVA|EFAVIRENZ,sustivatablets,sustiva,False,True
7,DPD_SANDOZ RIVASTIGMINE PATCH 5|RIVASTIGMINE,EMA_RIVASTIGMINE SANDOZ|RIVASTIGMINE,sandozrivastigminepatch5,rivastigminesandoz,False,True
8,DPD_IDELVION|COAGULATION FACTOR IX,EMA_IDELVION|ALBUTREPENONACOG ALFA,idelvion,idelvion,False,True
9,DPD_ALPROLIX|RECOMBINANT HUMAN COAGULATION FAC...,EMA_ALPROLIX|EFTRENONACOG ALFA,alprolix,alprolix,False,True


In [11]:
# Qwen top-5 candidates, already oriented as (DPD id, EMA id).
ranked = qwen_ranked[("DPD", "EMA")]
qwen5 = pd.MultiIndex.from_frame(ranked.loc[ranked["rank"].le(5), ["id1", "id2"]]).unique()

# Rebuild the prefix-4 candidate pairs from the full DPD and EMA records.
def prefix_frame(source, id_column):
    frame = brand_data[source][["id", "brand_key"]].rename(columns={"id": id_column})
    return frame.assign(prefix4=frame["brand_key"].str[:4])

prefix = prefix_frame("DPD", "id1").merge(prefix_frame("EMA", "id2"), on="prefix4")
prefix4 = pd.MultiIndex.from_frame(prefix[["id1", "id2"]]).unique()
combined = qwen5.union(prefix4, sort=False)

# `check` contains the 352 known DPD–EMA matches and their brand names.
gold_pairs = pd.MultiIndex.from_frame(check[["id1", "id2"]])
qwen_miss = ~gold_pairs.isin(qwen5)
prefix_hit = gold_pairs.isin(prefix4)
retained = int(gold_pairs.isin(combined).sum())

assert len(qwen5) == 13_240 and len(prefix4) == 16_398 and qwen_miss.sum() == 6

print(f"Unique candidates, Qwen top-5 OR prefix-4: {len(combined):,}")
print(f"Additional candidates over Qwen alone:     {len(combined) - len(qwen5):,}")
print(f"Qwen misses recovered by prefix-4:         {(qwen_miss & prefix_hit).sum()} of 6")
print(f"Known matches retained:                    {retained}/{len(check)} ({retained / len(check):.2%})")

display(
    check.loc[qwen_miss, ["id1", "id2", "DPD brand", "EMA brand"]]
    .assign(recovered_by_prefix4=prefix_hit[qwen_miss])
    .sort_values("recovered_by_prefix4", ascending=False)
    .reset_index(drop=True)
)

Unique candidates, Qwen top-5 OR prefix-4: 27,966
Additional candidates over Qwen alone:     14,726
Qwen misses recovered by prefix-4:         4 of 6
Known matches retained:                    350/352 (99.43%)


,id1,id2,DPD brand,EMA brand,recovered_by_prefix4
0,DPD_SUSTIVA TABLETS|EFAVIRENZ,EMA_SUSTIVA|EFAVIRENZ,sustivatablets,sustiva,True
1,DPD_IDELVION|COAGULATION FACTOR IX,EMA_IDELVION|ALBUTREPENONACOG ALFA,idelvion,idelvion,True
2,DPD_ALPROLIX|RECOMBINANT HUMAN COAGULATION FAC...,EMA_ALPROLIX|EFTRENONACOG ALFA,alprolix,alprolix,True
3,DPD_VIAGRA|SILDENAFIL,EMA_VIAGRA|SILDENAFIL,viagra,viagra,True
4,DPD_SANDOZ RIVASTIGMINE PATCH 5|RIVASTIGMINE,EMA_RIVASTIGMINE SANDOZ|RIVASTIGMINE,sandozrivastigminepatch5,rivastigminesandoz,False
5,DPD_MYLAN TADALAFIL|TADALAFIL,EMA_TADALAFIL MYLAN|TADALAFIL,mylantadalafil,tadalafilmylan,False


In [12]:
from itertools import combinations

rows = []
for left, right in combinations(("DPD", "EMA", "FDA"), 2):
    ranked = qwen_ranked[(left, right)]
    qwen5 = pd.MultiIndex.from_frame(
        ranked.loc[ranked["rank"].le(5), ["id1", "id2"]]
    ).unique()

    prefix = prefix_frame(left, "id1").merge(prefix_frame(right, "id2"), on="prefix4")
    prefix4 = pd.MultiIndex.from_frame(prefix[["id1", "id2"]]).unique()
    combined = qwen5.union(prefix4, sort=False)

    positives = dev_eval.loc[
        dev_eval["source1"].eq(left)
        & dev_eval["source2"].eq(right)
        & dev_eval["match"].astype(int).eq(1),
        ["id1", "id2"],
    ]
    gold_pairs = pd.MultiIndex.from_frame(positives)
    qwen_hits = int(gold_pairs.isin(qwen5).sum())
    possible = len(ids[left]) * len(ids[right])

    for method, candidates in [("Qwen top-5", qwen5), ("Qwen top-5 OR prefix-4", combined)]:
        retained = int(gold_pairs.isin(candidates).sum())
        rows.append({
            "Source pair": f"{left}–{right}",
            "Method": method,
            "Unique candidates": len(candidates),
            "Extra candidates vs Qwen": len(candidates) - len(qwen5),
            "Known matches": len(gold_pairs),
            "Retained": retained,
            "Missed": len(gold_pairs) - retained,
            "Extra matches vs Qwen": retained - qwen_hits,
            "Pair completeness": retained / len(gold_pairs),
            "Reduction ratio": 1 - len(candidates) / possible,
        })

qwen_prefix_results = pd.DataFrame(rows)
display(qwen_prefix_results.style.hide(axis="index").format({
    "Unique candidates": "{:,}",
    "Extra candidates vs Qwen": "{:,}",
    "Pair completeness": "{:.2%}",
    "Reduction ratio": "{:.4%}",
}))

Source pair,Method,Unique candidates,Extra candidates vs Qwen,Known matches,Retained,Missed,Extra matches vs Qwen,Pair completeness,Reduction ratio
DPD–EMA,Qwen top-5,"13,240",0,352,346,6,0,98.30%,99.9880%
DPD–EMA,Qwen top-5 OR prefix-4,"27,966","14,726",352,350,2,4,99.43%,99.9746%
DPD–FDA,Qwen top-5,"40,875",0,211,210,1,0,99.53%,99.9880%
DPD–FDA,Qwen top-5 OR prefix-4,"162,930","122,055",211,211,0,1,100.00%,99.9521%
EMA–FDA,Qwen top-5,"13,240",0,475,465,10,0,97.89%,99.9388%
EMA–FDA,Qwen top-5 OR prefix-4,"15,934","2,694",475,474,1,9,99.79%,99.9264%


## 3. Frozen evaluation of Qwen top-5

The development criterion selected top-5. Reuse the full-source shortlist, translate test EMA IDs with the same lookup, and exclude canonical pairs seen anywhere in development, including conflicts. Test labels enter the workflow for the first time here.

In [13]:
# Frozen blocker: local Qwen, top 5, same retrieval direction and rankings as development.
FROZEN_K = 5
pair_key = ["source1", "id1", "source2", "id2"]

test = pd.read_csv(gold_dir / "gs_pair_test.csv", dtype=str, encoding="utf-8-sig")
assert test["match"].isin(["0", "1"]).all()
assert test["hard"].isin(["0", "1"]).all()

# Preserve original IDs and apply the same team EMA lookup used for development.
for side in ("1", "2"):
    source, key = "source" + side, "id" + side
    test[key + "_original"] = test[key]
    needs_lookup = test[source].eq("EMA") & ~test[key].isin(ids["EMA"])
    test.loc[needs_lookup, key] = test.loc[needs_lookup, key].map(ema_ids)

resolved = pd.Series(
    [a in ids[s1] and b in ids[s2] for s1, a, s2, b in test[pair_key].itertuples(index=False, name=None)],
    index=test.index,
)
test_unresolved = test.loc[~resolved].copy()
valid = test.loc[resolved].copy()

# Set aside all rows of a conflicting pair. Keep one copy of each agreeing pair.
conflict_mask = valid.groupby(pair_key)["match"].transform("nunique").gt(1)
test_conflicts = valid.loc[conflict_mask].copy()
agreeing = valid.loc[~conflict_mask].sort_values("hard", ascending=False, kind="stable")
test_unique = agreeing.drop_duplicates(pair_key).copy()

# Exclude canonical pairs seen anywhere in development, including its quarantined conflicts.
dev_keys = pd.MultiIndex.from_frame(dev[pair_key]).unique()
overlap_mask = pd.MultiIndex.from_frame(test_unique[pair_key]).isin(dev_keys)
test_overlap = test_unique.loc[overlap_mask].copy()
test_eval = test_unique.loc[~overlap_mask].copy()

extra_copies = len(agreeing) - len(test_unique)
assert len(test) == (
    len(test_unresolved) + len(test_conflicts) + extra_copies + len(test_overlap) + len(test_eval)
)
print(f"Original test rows:                  {len(test)}")
print(f"Rows with unresolved IDs:            {len(test_unresolved)}")
print(f"Rows set aside for label conflicts:  {len(test_conflicts)}")
print(f"Extra agreeing copies removed:       {extra_copies}")
print(f"Distinct pairs seen in development:  {len(test_overlap)}")
print(f"Eligible held-out pairs:             {len(test_eval)}")

# Score the already-generated full-source top-5 shortlists against held-out positives.
rows = []
for (left, right), ranked in qwen_ranked.items():
    candidates = pd.MultiIndex.from_frame(
        ranked.loc[ranked["rank"].le(FROZEN_K), ["id1", "id2"]]
    ).unique()

    subset = test_eval.loc[test_eval["source1"].eq(left) & test_eval["source2"].eq(right)]
    positives = subset.loc[subset["match"].eq("1"), ["id1", "id2"]]
    retained = int(pd.MultiIndex.from_frame(positives).isin(candidates).sum())

    rows.append({
        "Source pair": f"{left}–{right}",
        "Eligible test pairs": len(subset),
        "Possible comparisons": len(ids[left]) * len(ids[right]),
        "Unique candidates": len(candidates),
        "Known test matches": len(positives),
        "Retained": retained,
        "Missed": len(positives) - retained,
    })

qwen_test_results = pd.DataFrame(rows).set_index("Source pair")
assert qwen_test_results["Eligible test pairs"].sum() == len(test_eval)
qwen_test_results.loc["All pairs"] = qwen_test_results.sum()
qwen_test_results["Pair completeness"] = (
    qwen_test_results["Retained"] / qwen_test_results["Known test matches"].replace(0, float("nan"))
)
qwen_test_results["Reduction ratio"] = (
    1 - qwen_test_results["Unique candidates"] / qwen_test_results["Possible comparisons"]
)

display(qwen_test_results.style.format({
    "Possible comparisons": "{:,}",
    "Unique candidates": "{:,}",
    "Pair completeness": "{:.2%}",
    "Reduction ratio": "{:.4%}",
}, na_rep="—"))

Original test rows:                  525
Rows with unresolved IDs:            0
Rows set aside for label conflicts:  0
Extra agreeing copies removed:       0
Distinct pairs seen in development:  4
Eligible held-out pairs:             521


,Eligible test pairs,Possible comparisons,Unique candidates,Known test matches,Retained,Missed,Pair completeness,Reduction ratio
Source pair,,,,,,,,
DPD–EMA,150,"110,252,128","13,240",63,62,1,98.41%,99.9880%
DPD–FDA,116,"340,374,300","40,875",37,36,1,97.30%,99.9880%
EMA–FDA,255,"21,647,400","13,240",85,79,6,92.94%,99.9388%
All pairs,521,"472,273,828","67,355",185,177,8,95.68%,99.9857%


### Frozen blocker evaluation

We selected Qwen top-5 because it had the smallest candidate set among the configurations that exceeded 97% pair completeness on every development source pair. It retained 1,021 of 1,038 known development matches (98.36%).

After EMA ID alignment, four of the 525 test pairs also appeared in development. We excluded those four, leaving 521 eligible held-out pairs, including 185 known matches.

| Source pair | Known test matches retained | Pair completeness | Above 97%? |
|---|---:|---:|---|
| DPD–EMA | 62/63 | 98.41% | Yes |
| DPD–FDA | 36/37 | 97.30% | Yes |
| EMA–FDA | 79/85 | 92.94% | No |
| **Pooled** | **177/185** | **95.68%** | **No** |

Qwen top-5 reduced 472,273,828 possible comparisons to 67,355 candidates. It missed the held-out completeness target, with six of the eight misses occurring on EMA–FDA.

## 4. Exploratory follow-up after seeing the frozen test result

EMA–FDA missed the target. Reuse the same 521 eligible pairs to assess the incremental contribution of a brand prefix and larger neighbor lists. These checks share the previously inspected test labels. The independent result for the originally frozen choice is 177/185.

In [14]:
rows = []

for (left, right), ranked in qwen_ranked.items():
    # Reuse the frozen Qwen rankings and the same four-character prefix rule.
    qwen5 = pd.MultiIndex.from_frame(
        ranked.loc[ranked["rank"].le(5), ["id1", "id2"]]
    ).unique()

    prefix = prefix_frame(left, "id1").merge(prefix_frame(right, "id2"), on="prefix4")
    prefix4 = pd.MultiIndex.from_frame(prefix[["id1", "id2"]]).unique()
    combined = qwen5.union(prefix4, sort=False)

    positives = test_eval.loc[
        test_eval["source1"].eq(left)
        & test_eval["source2"].eq(right)
        & test_eval["match"].astype(int).eq(1),
        ["id1", "id2"],
    ]
    gold_pairs = pd.MultiIndex.from_frame(positives)
    qwen_hit = gold_pairs.isin(qwen5)
    combined_hit = gold_pairs.isin(combined)

    rows.append({
        "Source pair": f"{left}–{right}",
        "Union candidates": len(combined),
        "Extra candidates": len(combined) - len(qwen5),
        "Known matches": len(gold_pairs),
        "Qwen retained": int(qwen_hit.sum()),
        "Recovered by prefix": int((~qwen_hit & combined_hit).sum()),
        "Union retained": int(combined_hit.sum()),
        "Still missed": int((~combined_hit).sum()),
    })

exploratory_test = pd.DataFrame(rows).set_index("Source pair")
exploratory_test.loc["Pooled"] = exploratory_test.sum()

# Verify that this comparison reproduces the frozen baseline on the same positives.
assert exploratory_test.loc["Pooled", "Known matches"] == 185
assert exploratory_test.loc["Pooled", "Qwen retained"] == 177

exploratory_test["Qwen PC"] = exploratory_test["Qwen retained"] / exploratory_test["Known matches"]
exploratory_test["Union PC"] = exploratory_test["Union retained"] / exploratory_test["Known matches"]
exploratory_test["Union above 97%?"] = exploratory_test["Union PC"] > 0.97

display(exploratory_test.style.format({
    "Union candidates": "{:,}",
    "Extra candidates": "{:,}",
    "Qwen PC": "{:.2%}",
    "Union PC": "{:.2%}",
}))

,Union candidates,Extra candidates,Known matches,Qwen retained,Recovered by prefix,Union retained,Still missed,Qwen PC,Union PC,Union above 97%?
Source pair,,,,,,,,,,
DPD–EMA,"27,966","14,726",63,62,1,63,0,98.41%,100.00%,True
DPD–FDA,"162,930","122,055",37,36,1,37,0,97.30%,100.00%,True
EMA–FDA,"15,934","2,694",85,79,4,83,2,92.94%,97.65%,True
Pooled,"206,830","139,475",185,177,6,183,2,95.68%,98.92%,True


### Increasing the neighbor count

Compare ranks 6–10 and 11–20 with the frozen top-5 shortlist.

In [15]:
rows = []
for (left, right), ranked in qwen_ranked.items():
    positives = test_eval.loc[
        test_eval["source1"].eq(left)
        & test_eval["source2"].eq(right)
        & test_eval["match"].astype(int).eq(1),
        ["id1", "id2"],
    ]
    gold_pairs = pd.MultiIndex.from_frame(positives)

    row = {"Source pair": f"{left}–{right}", "Known matches": len(gold_pairs)}
    for k in (5, 10):
        candidates = pd.MultiIndex.from_frame(
            ranked.loc[ranked["rank"].le(k), ["id1", "id2"]]
        ).unique()
        row[f"Candidates @{k}"] = len(candidates)
        row[f"Retained @{k}"] = int(gold_pairs.isin(candidates).sum())
    rows.append(row)

top10_test = pd.DataFrame(rows).set_index("Source pair")
top10_test.loc["Pooled"] = top10_test.sum()
assert top10_test.loc["Pooled", "Retained @5"] == 177

top10_test["Extra candidates"] = top10_test["Candidates @10"] - top10_test["Candidates @5"]
top10_test["Recovered at ranks 6–10"] = top10_test["Retained @10"] - top10_test["Retained @5"]
top10_test["Still missed"] = top10_test["Known matches"] - top10_test["Retained @10"]
top10_test["PC @10"] = top10_test["Retained @10"] / top10_test["Known matches"]
top10_test["Above 97%?"] = top10_test["PC @10"] > 0.97

display(top10_test.style.format({
    "Candidates @5": "{:,}",
    "Candidates @10": "{:,}",
    "Extra candidates": "{:,}",
    "PC @10": "{:.2%}",
}))

,Known matches,Candidates @5,Retained @5,Candidates @10,Retained @10,Extra candidates,Recovered at ranks 6–10,Still missed,PC @10,Above 97%?
Source pair,,,,,,,,,,
DPD–EMA,63,"13,240",62,"26,480",62,"13,240",0,1,98.41%,True
DPD–FDA,37,"40,875",36,"81,750",37,"40,875",1,0,100.00%,True
EMA–FDA,85,"13,240",79,"26,480",81,"13,240",2,4,95.29%,False
Pooled,185,"67,355",177,"134,710",180,"67,355",3,5,97.30%,True


In [16]:
rows = []
for (left, right), ranked in qwen_ranked.items():
    positives = test_eval.loc[
        test_eval["source1"].eq(left)
        & test_eval["source2"].eq(right)
        & test_eval["match"].astype(int).eq(1),
        ["id1", "id2"],
    ]

    # A missing rank means the known match was outside the cached top 20.
    ranks = positives.merge(
        ranked[["id1", "id2", "rank"]],
        on=["id1", "id2"], how="left", validate="one_to_one",
    )["rank"]

    rows.append({
        "Source pair": f"{left}–{right}",
        "Known matches": len(ranks),
        "Top-20 candidates": len(ranked),
        "Retained @10": int(ranks.le(10).sum()),
        "Recovered at ranks 11–20": int(ranks.between(11, 20).sum()),
        "Retained @20": int(ranks.le(20).sum()),
        "Still missed": int(ranks.isna().sum()),
    })

top20_test = pd.DataFrame(rows).set_index("Source pair")
top20_test.loc["Pooled"] = top20_test.sum()
assert top20_test.loc["Pooled", "Retained @10"] == 180

top20_test["PC @20"] = top20_test["Retained @20"] / top20_test["Known matches"]
top20_test["Above 97%?"] = top20_test["PC @20"] > 0.97
display(top20_test.style.format({"Top-20 candidates": "{:,}", "PC @20": "{:.2%}"}))

,Known matches,Top-20 candidates,Retained @10,Recovered at ranks 11–20,Retained @20,Still missed,PC @20,Above 97%?
Source pair,,,,,,,,
DPD–EMA,63,"52,960",62,0,62,1,98.41%,True
DPD–FDA,37,"163,500",37,0,37,0,100.00%,True
EMA–FDA,85,"52,960",81,1,82,3,96.47%,False
Pooled,185,"269,420",180,1,181,4,97.84%,True


### Marginal value of a third pass

Measure the extra trigram candidates and which development positives they recover. Then inspect the two labeled EMA–FDA test positives the two-pass union missed. These are post-selection diagnostics.

In [17]:
rows = []
miss_details = []

for (left, right), ranked in qwen_ranked.items():
    # Baseline: Qwen top-5 OR prefix-4.
    qwen5 = pd.MultiIndex.from_frame(
        ranked.loc[ranked["rank"].le(5), ["id1", "id2"]]
    ).unique()
    prefix = prefix_frame(left, "id1").merge(prefix_frame(right, "id2"), on="prefix4")
    prefix4 = pd.MultiIndex.from_frame(prefix[["id1", "id2"]]).unique()
    baseline = qwen5.union(prefix4, sort=False)

    # Rebuild the same cap-1,000 trigram rule for this source pair.
    left_freq, right_freq = frequencies[left], frequencies[right]
    allowed = {
        t for t in left_freq.keys() & right_freq.keys()
        if left_freq[t] * right_freq[t] <= 1000
    }
    blocker = TokenBlocker(
        brand_data[left], brand_data[right],
        column="brand_key",
        id_column="id",
        tokenizer=lambda name: sorted(brand_trigrams(name) & allowed),
        output_dir=str(repo / "output" / "blocking" / f"{left}_{right}" / "trigram1000"),
    )
    trigrams = candidate_index(blocker)
    extra = trigrams.difference(baseline, sort=False)

    # Score only against development positives.
    positives = dev_eval.loc[
        dev_eval["source1"].eq(left)
        & dev_eval["source2"].eq(right)
        & dev_eval["match"].astype(int).eq(1),
        ["id1", "id2"],
    ]
    gold_pairs = pd.MultiIndex.from_frame(positives)
    missed = positives.loc[~gold_pairs.isin(baseline)].copy()
    missed["Recovered by trigrams"] = pd.MultiIndex.from_frame(missed).isin(extra)
    missed["Source pair"] = f"{left}–{right}"
    miss_details.append(missed)

    recovered = int(missed["Recovered by trigrams"].sum())
    retained = len(positives) - len(missed) + recovered
    rows.append({
        "Source pair": f"{left}–{right}",
        "Baseline candidates": len(baseline),
        "Extra unique candidates": len(extra),
        "Three-way union candidates": len(baseline) + len(extra),
        "Known dev matches": len(positives),
        "Baseline misses": len(missed),
        "Recovered by trigrams": recovered,
        "Still missed": len(missed) - recovered,
        "Retained by three-way union": retained,
    })

trigram_gain = pd.DataFrame(rows).set_index("Source pair")
trigram_gain.loc["Pooled"] = trigram_gain.sum()
assert trigram_gain.loc["Pooled", "Baseline misses"] == 3
trigram_gain["Union completeness"] = (
    trigram_gain["Retained by three-way union"] / trigram_gain["Known dev matches"]
)

display(trigram_gain.style.format({
    "Baseline candidates": "{:,}",
    "Extra unique candidates": "{:,}",
    "Three-way union candidates": "{:,}",
    "Union completeness": "{:.2%}",
}))
display(pd.concat(miss_details, ignore_index=True)[
    ["Source pair", "id1", "id2", "Recovered by trigrams"]
])

,Baseline candidates,Extra unique candidates,Three-way union candidates,Known dev matches,Baseline misses,Recovered by trigrams,Still missed,Retained by three-way union,Union completeness
Source pair,,,,,,,,,
DPD–EMA,"27,966","425,464","453,430",352,2,2,0,352,100.00%
DPD–FDA,"162,930","527,447","690,377",211,0,0,0,211,100.00%
EMA–FDA,"15,934","256,442","272,376",475,1,1,0,475,100.00%
Pooled,"206,830","1,209,353","1,416,183",1038,3,3,0,1038,100.00%


,Source pair,id1,id2,Recovered by trigrams
0,DPD–EMA,DPD_SANDOZ RIVASTIGMINE PATCH 5|RIVASTIGMINE,EMA_RIVASTIGMINE SANDOZ|RIVASTIGMINE,True
1,DPD–EMA,DPD_MYLAN TADALAFIL|TADALAFIL,EMA_TADALAFIL MYLAN|TADALAFIL,True
2,EMA–FDA,EMA_ENRYLAZE|CRISANTASPASE,FDA_RYLAZE|ASPARAGINASE ERWINIA CHRYSANTHEMI R...,True


In [18]:
# Exploratory: check the two EMA–FDA test matches missed by Qwen top-5 OR prefix-4.
pair_cols = ["id1", "id2"]
ranked = qwen_ranked[("EMA", "FDA")]
qwen5 = pd.MultiIndex.from_frame(ranked.loc[ranked["rank"].le(5), pair_cols])

positives = test_eval.loc[
    test_eval["source1"].eq("EMA")
    & test_eval["source2"].eq("FDA")
    & test_eval["match"].astype(int).eq(1),
    pair_cols,
].copy()

ema_keys = brand_data["EMA"].set_index("id")["brand_key"]
fda_keys = brand_data["FDA"].set_index("id")["brand_key"]
positives["EMA brand"] = positives["id1"].map(ema_keys)
positives["FDA brand"] = positives["id2"].map(fda_keys)
assert positives[["EMA brand", "FDA brand"]].notna().all().all()

qwen_hit = pd.MultiIndex.from_frame(positives[pair_cols]).isin(qwen5)
prefix_hit = positives["EMA brand"].str[:4].eq(positives["FDA brand"].str[:4])
misses = positives.loc[~(qwen_hit | prefix_hit)].copy()
assert len(positives) == 85 and len(misses) == 2

# Apply the same cap using frequencies from the FULL EMA and FDA datasets.
ema_freq, fda_freq = frequencies["EMA"], frequencies["FDA"]
allowed = {
    t for t in ema_freq.keys() & fda_freq.keys()
    if ema_freq[t] * fda_freq[t] <= 1000
}

misses["Retained shared trigrams"] = [
    sorted(brand_trigrams(ema) & brand_trigrams(fda) & allowed)
    for ema, fda in zip(misses["EMA brand"], misses["FDA brand"])
]
misses["Recovered by trigrams"] = misses["Retained shared trigrams"].map(bool)

recovered = int(misses["Recovered by trigrams"].sum())
retained = len(positives) - len(misses) + recovered
print(f"Recovered by trigrams: {recovered} of {len(misses)}")
print(f"EMA–FDA test completeness: {retained}/{len(positives)} ({retained / len(positives):.2%})")
display(misses.reset_index(drop=True))

Recovered by trigrams: 2 of 2
EMA–FDA test completeness: 85/85 (100.00%)


,id1,id2,EMA brand,FDA brand,Retained shared trigrams,Recovered by trigrams
0,EMA_COPALIA HCT|AMLODIPINE; HYDROCHLOROTHIAZID...,FDA_EXFORGE HCT|AMLODIPINE BESYLATE; HYDROCHLO...,copaliahct,exforgehct,[hct],True
1,EMA_IVEMEND|FOSAPREPITANT,FDA_EMEND|APREPITANT,ivemend,emend,"[end, men]",True


## 5. Handoff decision

Export **Qwen top-5 OR prefix-4**, deduplicated by the two XML IDs. Its 206,830 candidates retained 1,035/1,038 development positives and 183/185 positives in the exploratory test comparison. Top-20 alone produced 269,420 candidates and retained 181/185. Adding a trigram pass brought the development and exploratory test counts to 1,038/1,038 and 185/185, while increasing the handoff to 1,416,183 candidates. All scores use the original reviewed labels.

The course's 97% guideline is applied per source pair here. Downstream matcher precision, recall, F1, accepted correspondences and end-to-end time require a separate matcher evaluation.

### Export and verify candidate pairs

The matcher needs three gzip CSVs (`source_pair,id1,id2`), the matching XML revision, and the manifest written beside them. Join the IDs to the corresponding source XML to obtain matcher inputs. A candidate row carries no match label. File order and logical row counts are checked on read-back.

**Sharing embeddings.** The matcher can consume the CSVs directly. To reproduce retrieval, regenerate vectors from the pinned model or share the optional `.npy` cache in an artifact store or release. Copy it under the revision directory in `WDI_EMBEDDING_CACHE` (default: `.embedding-cache` next to the repository). Matrix rows follow XML medicine order; the cache filenames fingerprint the ordered IDs and input texts. Compare SHA-256 values with the manifest before reusing shared matrices.

In [19]:
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path
from tempfile import TemporaryDirectory

import json
import platform
import subprocess
from importlib.metadata import version

def sha256_file(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


expected_counts = {
    ("DPD", "EMA"): 27_966,
    ("DPD", "FDA"): 162_930,
    ("EMA", "FDA"): 15_934,
}

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
handoff_root = repo / "output" / "blocking" / "handoff"
handoff_root.mkdir(parents=True, exist_ok=True)
handoff_dir = handoff_root / f"qwen_top5_or_prefix4_{run_id}"
summary = []

# Publish the handoff only after all three files pass verification.
with TemporaryDirectory(dir=handoff_root) as tmp:
    stage = Path(tmp) / "export"
    stage.mkdir()

    for left, right in combinations(("DPD", "EMA", "FDA"), 2):
        pair_name = f"{left.lower()}_{right.lower()}"
        ranked = qwen_ranked[(left, right)]
        qwen5 = ranked.loc[ranked["rank"].le(5), ["id1", "id2"]]

        prefix4 = prefix_frame(left, "id1").merge(
            prefix_frame(right, "id2"), on="prefix4"
        )[["id1", "id2"]]

        candidates = (
            pd.concat([qwen5, prefix4], ignore_index=True)
            .drop_duplicates()
            .sort_values(["id1", "id2"], ignore_index=True)
        )
        candidates.insert(0, "source_pair", pair_name)

        filename = f"{pair_name}_candidates.csv.gz"
        path = stage / filename
        candidates.to_csv(path, index=False, compression={"method": "gzip", "mtime": 0})

        # Verify the exported file itself: contents, count, uniqueness and source IDs.
        saved = pd.read_csv(path, dtype=str, keep_default_na=False)
        pd.testing.assert_frame_equal(saved, candidates, check_dtype=False)
        assert len(saved) == expected_counts[(left, right)], f"{pair_name}: unexpected count"
        assert not saved.duplicated(["id1", "id2"]).any(), f"{pair_name}: duplicate pair"
        assert saved["id1"].isin(ids[left]).all(), f"{pair_name}: invalid {left} ID"
        assert saved["id2"].isin(ids[right]).all(), f"{pair_name}: invalid {right} ID"

        summary.append({"Source pair": pair_name, "Verified pairs": len(saved), "File": filename})

    handoff_summary = pd.DataFrame(summary)
    assert handoff_summary["Verified pairs"].sum() == 206_830
    input_paths = [xml_dir / f"{s.lower()}_medicines.xml" for s in ("DPD", "EMA", "FDA")]
    input_paths += [repo / "data/2_processed/ema_entities.csv", gold_dir / "gs_pair_train_val.csv",
                    gold_dir / "gs_pair_test.csv", repo / "src/golden_standard/gold_standard_pair_gen.py"]
    manifest = {
        "data_checkout": subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip(),
        "rule": "union of Qwen cosine top-5 (smaller source queries larger) and normalized brand prefix-4",
        "model": MODEL_ID, "model_revision": MODEL_REVISION, "input": "brand and sorted active substances",
        "search": {"backend": "sklearn", "metric": "cosine", "retrieved": 20, "selected": 5, "minimum_similarity": 0.0},
        "python": platform.python_version(),
        "packages": {p: version(p) for p in ("uma-pydi", "pandas", "numpy", "scikit-learn", "torch",
                                              "sentence-transformers", "transformers", "huggingface-hub")},
        "input_sha256": {str(path.relative_to(repo)): sha256_file(path) for path in input_paths},
        "vectors": {s: {"file": path.name, "shape": list(vectors[s].shape), "sha256": sha256_file(path)}
                    for s, path in vector_paths.items()},
        "exports": [{**row, "sha256": sha256_file(stage / row["File"])} for row in summary],
    }
    (stage / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    stage.rename(handoff_dir)

display(handoff_summary.style.hide(axis="index").format({"Verified pairs": "{:,}"}))
print(f"Verified total: {handoff_summary['Verified pairs'].sum():,} unique candidate pairs")
print(f"Handoff folder: {handoff_dir.relative_to(repo)}")

Source pair,Verified pairs,File
dpd_ema,"27,966",dpd_ema_candidates.csv.gz
dpd_fda,"162,930",dpd_fda_candidates.csv.gz
ema_fda,"15,934",ema_fda_candidates.csv.gz


Verified total: 206,830 unique candidate pairs
Handoff folder: output/blocking/handoff/qwen_top5_or_prefix4_20261110T192842809858Z
